In [ ]:
# Standard library
import os
import random
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Torchvision
import torchvision
import torchvision.transforms as transforms

# Progress bars
from tqdm import tqdm

# Experiment tracking
import wandb


plt.style.use("default")
import sys
from dataclasses import dataclass

In [ ]:
# 1. Clone your clean repository code from GitHub
REPO_URL = "https://github.com/K0NSTANT1N3/Facial-Expression-Recognition.git"
REPO_NAME = "Facial-Expression-Recognition"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repository already exists. Pulling latest updates...")
    os.system(f"cd {REPO_NAME} && git pull")

# 2. THE SECRET SAUCE: Add your cloned repo directory to Python's path
sys.path.append(os.path.abspath(REPO_NAME))

print("Successfully linked GitHub repository and adjusted system paths!")


In [ ]:
import wandb

wandb.login(key="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")

In [ ]:
import wandb

run = wandb.init(
    project="fer2013",
    entity="kende23-n-a",
    name="test_run"
)

for epoch in range(5):
    run.log({
        "epoch": epoch,
        "accuracy": epoch * 0.1,
        "loss": 1/(epoch+1)
    })

run.finish()

In [ ]:
@dataclass
class Config:
    csv_path: str = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 10

    num_classes: int = 7
    image_size: int = 48

    random_seed: int = 42


config = Config()

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.random_seed)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
df = pd.read_csv(config.csv_path)

print(df.shape)
df.head()

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=config.random_seed,
    stratify=df["emotion"]
)

print(len(train_df))
print(len(val_df))

In [ ]:
def pixels_to_image(pixel_string):
    pixels = np.fromstring(
        pixel_string,
        dtype=np.uint8,
        sep=" "
    )

    return pixels.reshape(48, 48)

In [ ]:
class FERDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = pixels_to_image(row["pixels"])

        image = image.astype(np.float32) / 255.0

        image = torch.tensor(image).unsqueeze(0)

        label = torch.tensor(
            row["emotion"],
            dtype=torch.long
        )

        return image, label

In [ ]:
train_dataset = FERDataset(train_df)
val_dataset = FERDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),

            nn.Linear(128, 7)
        )

    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(x, start_dim=1)

        x = self.classifier(x)

        return x

In [ ]:
model = SimpleCNN().to(device)

print(model)

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.lr
)

In [ ]:
wandb.init(
    project="fer2013",
    entity="kende23-n-a",
    name="baseline_cnn",
    config=vars(config)
)

wandb.watch(model)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() #?

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [ ]:
print(next(model.parameters()).device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
best_val_acc = 0

for epoch in range(config.epochs):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch+1}/{config.epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_baseline_model.pt"
        )

In [ ]:
wandb.finish()